### Importando as bibliotecas e criação da função que converte os números do padrão brasileiro para o padrão do bano de dados

In [1]:
import pdfplumber
import pandas as pd
import os


In [2]:

def limpar_numero(valor):

    # Trata valores ausentes
    if valor is None:
        return None

    # Se já for numérico, não altera o valor
    if isinstance(valor, (int, float)):
        return float(valor)

    valor_str = str(valor).strip()

    # Trata campos vazios ou representados por traço
    if valor_str in ("", "-"):
        return None

    # Converte o padrão brasileiro para o padrão decimal do Python
    valor_str = valor_str.replace(".", "").replace(",", ".")

    try:
        return float(valor_str)
    except ValueError:
        return None


### testando a função

In [6]:

# Casos de teste
valores_teste = [
    "5,38",
    "1.234,56",
    "0,00",
    "120",
    "-",
    "",
    None,
    "valor inválido",
    5.38
]

for valor in valores_teste:
    resultado = limpar_numero(valor)
    print(f"Entrada: {valor!r:18} | Saída: {resultado!r}")


Entrada: '5,38'             | Saída: 5.38
Entrada: '1.234,56'         | Saída: 1234.56
Entrada: '0,00'             | Saída: 0.0
Entrada: '120'              | Saída: 120.0
Entrada: '-'                | Saída: None
Entrada: ''                 | Saída: None
Entrada: None               | Saída: None
Entrada: 'valor inválido'   | Saída: None
Entrada: 5.38               | Saída: 5.38


#### revisão da extração do PDF

In [10]:
caminho_teste = "CandidatoVaga/candidato-vaga-2018.pdf"

with pdfplumber.open(caminho_teste) as pdf:
    pagina_teste = pdf.pages[0]
    tabela_bruta = pagina_teste.extract_table()
    print(f"Total de linhas extraídas na página 1: {len(tabela_bruta)}\n")
    print(tabela_bruta[0])
    print(tabela_bruta[1])

Total de linhas extraídas na página 1: 23

['Administração (RIO)', '66', '355', '5,38', '24', '24', '1,00', '24', '15', '0,63', '6', '0', '0,00', '120', '394', '3,28']
['Arqueologia (RIO)', '16', '58', '3,63', '6', '1', '0,17', '6', '3', '0,50', '2', '0', '0,00', '30', '62', '2,07']


In [11]:
with pdfplumber.open(caminho_teste) as pdf:

    print(f"Total de páginas do PDF: {len(pdf.pages)}")

    for numero_pagina, pagina in enumerate(pdf.pages[:3], start=1):

        tabela_bruta = pagina.extract_table()

        print(f"\n--- Página {numero_pagina} ---")

        if not tabela_bruta:
            print("Nenhuma tabela identificada.")
            continue

        print(f"Total de linhas: {len(tabela_bruta)}")

        for numero_linha, linha in enumerate(tabela_bruta[:5], start=1):
            print(f"Linha {numero_linha}: {linha}")
            print(f"Quantidade de colunas: {len(linha)}")

Total de páginas do PDF: 3

--- Página 1 ---
Total de linhas: 23
Linha 1: ['Administração (RIO)', '66', '355', '5,38', '24', '24', '1,00', '24', '15', '0,63', '6', '0', '0,00', '120', '394', '3,28']
Quantidade de colunas: 16
Linha 2: ['Arqueologia (RIO)', '16', '58', '3,63', '6', '1', '0,17', '6', '3', '0,50', '2', '0', '0,00', '30', '62', '2,07']
Quantidade de colunas: 16
Linha 3: ['Arquitetura e Urbanismo\n(PETRÓPOLIS)', '26', '348', '13,38', '10', '9', '0,90', '10', '6', '0,60', '4', '0', '0,00', '50', '363', '7,26']
Quantidade de colunas: 16
Linha 4: ['Artes Visuais (RIO)', '38', '174', '4,58', '14', '2', '0,14', '14', '3', '0,21', '4', '0', '0,00', '70', '179', '2,56']
Quantidade de colunas: 16
Linha 5: ['Ciência da Computação\n(RIO)', '66', '433', '6,56', '24', '15', '0,63', '24', '8', '0,33', '6', '1', '0,17', '120', '457', '3,81']
Quantidade de colunas: 16

--- Página 2 ---
Total de linhas: 22
Linha 1: ['Engenharia Mecânica\n(RESENDE)', '54', '69', '1,28', '20', '1', '0,05', '2

### EXTRAÇÃO

In [12]:
def extrair_vagas_dicionario(caminho_pdf, ano):

    dados_finais = []

    categorias = {
        "Não Reservada": (1, 2, 3),
        "Rede Pública": (4, 5, 6),
        "Negros/Indígenas": (7, 8, 9),
        "Pessoas com Deficiência": (10, 11, 12),
        "Geral": (13, 14, 15)
    }

    with pdfplumber.open(caminho_pdf) as pdf:

        for numero_pagina, pagina in enumerate(pdf.pages, start=1):

            tabela = pagina.extract_table()

            if not tabela:
                continue

            for numero_linha, linha in enumerate(tabela, start=1):

                if not linha or not linha[0]:
                    continue

                curso_nome = str(linha[0]).replace("\n", " ").strip()

                if "TOTAL" in curso_nome.upper():
                    continue

                # Valida a estrutura da linha
                if len(linha) != 16:
                    raise ValueError(
                        f"Estrutura inesperada no PDF: "
                        f"página {numero_pagina}, "
                        f"linha {numero_linha}. "
                        f"Esperadas 16 colunas, encontradas {len(linha)}."
                    )

                for categoria, indices in categorias.items():

                    indice_vagas, indice_inscritos, indice_relacao = indices

                    dados_finais.append({
                        "ano_vestibular": int(ano),
                        "curso": curso_nome,
                        "categoria_cota": categoria,
                        "vagas": limpar_numero(linha[indice_vagas]),
                        "inscritos": limpar_numero(linha[indice_inscritos]),
                        "relacao_cand_vaga": limpar_numero(linha[indice_relacao])
                    })

    return pd.DataFrame(dados_finais)

In [13]:
# teste da função:

df_2018 = extrair_vagas_dicionario(
    caminho_teste,
    2018
)

display(df_2018.head(10))

print("\nInformações do DataFrame:")
df_2018.info()

print("\nCategorias encontradas:")
print(df_2018["categoria_cota"].value_counts())

print("\nTotal de registros:")
print(len(df_2018))

,ano_vestibular,curso,categoria_cota,vagas,inscritos,relacao_cand_vaga
0,2018,Administração (RIO),Não Reservada,66.0,355.0,5.38
1,2018,Administração (RIO),Rede Pública,24.0,24.0,1.00
2,2018,Administração (RIO),Negros/Indígenas,24.0,15.0,0.63
3,2018,Administração (RIO),Pessoas com Deficiência,6.0,0.0,0.00
4,2018,Administração (RIO),Geral,120.0,394.0,3.28
5,2018,Arqueologia (RIO),Não Reservada,16.0,58.0,3.63
6,2018,Arqueologia (RIO),Rede Pública,6.0,1.0,0.17
7,2018,Arqueologia (RIO),Negros/Indígenas,6.0,3.0,0.50
8,2018,Arqueologia (RIO),Pessoas com Deficiência,2.0,0.0,0.00
9,2018,Arqueologia (RIO),Geral,30.0,62.0,2.07



Informações do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ano_vestibular     320 non-null    int64  
 1   curso              320 non-null    str    
 2   categoria_cota     320 non-null    str    
 3   vagas              320 non-null    float64
 4   inscritos          320 non-null    float64
 5   relacao_cand_vaga  320 non-null    float64
dtypes: float64(3), int64(1), str(2)
memory usage: 15.1 KB

Categorias encontradas:
categoria_cota
Não Reservada              64
Rede Pública               64
Negros/Indígenas           64
Pessoas com Deficiência    64
Geral                      64
Name: count, dtype: int64

Total de registros:
320


In [14]:
pasta_vagas = "CandidatoVaga"

# Localiza e ordena os arquivos PDF
arquivos_pdf = sorted([
    arquivo
    for arquivo in os.listdir(pasta_vagas)
    if arquivo.lower().endswith(".pdf")
])

if not arquivos_pdf:
    raise FileNotFoundError(
        f"Nenhum arquivo PDF encontrado na pasta: {pasta_vagas}"
    )

lista_dfs = []

print(f"Iniciando processamento de {len(arquivos_pdf)} arquivos...\n")

for arquivo in arquivos_pdf:

    print(f"Extraindo dados de: {arquivo}")

    caminho_completo = os.path.join(pasta_vagas, arquivo)

    # Extrai o ano dos últimos quatro caracteres do nome
    ano = arquivo[-8:-4]

    if not ano.isdigit():
        raise ValueError(
            f"Não foi possível identificar o ano do arquivo: {arquivo}"
        )

    df_ano = extrair_vagas_dicionario(
        caminho_completo,
        ano
    )

    if df_ano.empty:
        raise ValueError(
            f"Nenhum registro foi extraído do arquivo: {arquivo}"
        )

    print(f"Registros extraídos: {len(df_ano)}\n")

    lista_dfs.append(df_ano)

# Consolida os DataFrames
df_candidato_vaga = pd.concat(
    lista_dfs,
    ignore_index=True
)

print("Processamento concluído!")

df_candidato_vaga.info()

Iniciando processamento de 9 arquivos...

Extraindo dados de: candidato-vaga-2018.pdf
Registros extraídos: 320

Extraindo dados de: candidato-vaga-2019.pdf
Registros extraídos: 320

Extraindo dados de: candidato-vaga-2020.pdf
Registros extraídos: 320

Extraindo dados de: candidato-vaga-2021.pdf
Registros extraídos: 320

Extraindo dados de: candidato-vaga-2022.pdf
Registros extraídos: 320

Extraindo dados de: candidato-vaga-2023.pdf
Registros extraídos: 360

Extraindo dados de: candidato-vaga-2024.pdf
Registros extraídos: 380

Extraindo dados de: candidato-vaga-2025.pdf
Registros extraídos: 385

Extraindo dados de: candidato-vaga-2026.pdf
Registros extraídos: 395

Processamento concluído!
<class 'pandas.DataFrame'>
RangeIndex: 3120 entries, 0 to 3119
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ano_vestibular     3120 non-null   int64  
 1   curso              3120 non-null   str    
 2   categoria_c

### verificações 

In [15]:
print("Total de registros:", len(df_candidato_vaga))

print("\nRegistros por ano:")
print(df_candidato_vaga["ano_vestibular"].value_counts().sort_index())

print("\nRegistros por categoria:")
print(df_candidato_vaga["categoria_cota"].value_counts())

print("\nValores ausentes:")
print(df_candidato_vaga.isnull().sum())

print("\nCursos por ano:")
print(
    df_candidato_vaga
    .groupby("ano_vestibular")["curso"]
    .nunique()
)

Total de registros: 3120

Registros por ano:
ano_vestibular
2018    320
2019    320
2020    320
2021    320
2022    320
2023    360
2024    380
2025    385
2026    395
Name: count, dtype: int64

Registros por categoria:
categoria_cota
Não Reservada              624
Rede Pública               624
Negros/Indígenas           624
Pessoas com Deficiência    624
Geral                      624
Name: count, dtype: int64

Valores ausentes:
ano_vestibular       0
curso                0
categoria_cota       0
vagas                0
inscritos            0
relacao_cand_vaga    0
dtype: int64

Cursos por ano:
ano_vestibular
2018    64
2019    64
2020    64
2021    64
2022    64
2023    72
2024    76
2025    77
2026    79
Name: curso, dtype: int64


### validação da qualidade dos dados

In [16]:
# 1. Verificar registros duplicados
duplicados = df_candidato_vaga.duplicated(
    subset=["ano_vestibular", "curso", "categoria_cota"]
)

print("Registros duplicados:", duplicados.sum())


# 2. Verificar valores ausentes
print("\nValores ausentes por coluna:")
print(df_candidato_vaga.isnull().sum())


# 3. Verificar valores negativos
colunas_numericas = [
    "vagas",
    "inscritos",
    "relacao_cand_vaga"
]

print("\nValores negativos:")
for coluna in colunas_numericas:
    quantidade = (df_candidato_vaga[coluna] < 0).sum()
    print(f"{coluna}: {quantidade}")


# 4. Verificar se todos os cursos possuem as cinco categorias
categorias_por_curso = (
    df_candidato_vaga
    .groupby(["ano_vestibular", "curso"])["categoria_cota"]
    .nunique()
)

cursos_incompletos = categorias_por_curso[
    categorias_por_curso != 5
]

print("\nCursos sem as cinco categorias:")
print(cursos_incompletos)


# 5. Verificar relação candidato/vaga
df_validacao = df_candidato_vaga.copy()

df_validacao["relacao_calculada"] = (
    df_validacao["inscritos"] /
    df_validacao["vagas"].replace(0, float("nan"))
)

df_validacao["diferenca_relacao"] = (
    df_validacao["relacao_cand_vaga"] -
    df_validacao["relacao_calculada"]
).abs()

inconsistencias_relacao = df_validacao[
    df_validacao["diferenca_relacao"] > 0.02
]

print("\nInconsistências na relação candidato/vaga:")
print(len(inconsistencias_relacao))

display(inconsistencias_relacao.head(10))

Registros duplicados: 0

Valores ausentes por coluna:
ano_vestibular       0
curso                0
categoria_cota       0
vagas                0
inscritos            0
relacao_cand_vaga    0
dtype: int64

Valores negativos:
vagas: 0
inscritos: 0
relacao_cand_vaga: 0

Cursos sem as cinco categorias:
Series([], Name: categoria_cota, dtype: int64)

Inconsistências na relação candidato/vaga:
0


,ano_vestibular,curso,categoria_cota,vagas,inscritos,relacao_cand_vaga,relacao_calculada,diferenca_relacao


### inicio da tabela Candidato / Vaga

In [19]:
df_candidato_vaga.head()

,ano_vestibular,curso,categoria_cota,vagas,inscritos,relacao_cand_vaga
0,2018,Administração (RIO),Não Reservada,66.0,355.0,5.38
1,2018,Administração (RIO),Rede Pública,24.0,24.0,1.00
2,2018,Administração (RIO),Negros/Indígenas,24.0,15.0,0.63
3,2018,Administração (RIO),Pessoas com Deficiência,6.0,0.0,0.00
4,2018,Administração (RIO),Geral,120.0,394.0,3.28
